# 🧞 Step 8: Genie — Natural Language Alert Querying

SOC managers ask questions in plain English; Genie writes and runs the SQL against Unity Catalog.

**Genie space:** `SOC Alert Triage` (space ID below). Its full definition — tables, instructions,
example SQL, sample questions and benchmarks — is version-controlled in
[`genie/soc_alert_triage_space.json`](../genie/soc_alert_triage_space.json) and can be recreated with
`POST /api/2.0/genie/spaces`.

| Tables in the space | Purpose |
|---|---|
| `gold_enriched_alerts` | One row per alert: customer, source, risk tier/score, threat intel, MITRE, ground-truth label |
| `gold_alert_predictions` | ML-predicted severity + confidence (join on `event_id`) |
| `gold_investigation_summaries` | AI investigation summaries (join on `event_id`) |
| `analyst_actions` | Analyst triage decisions from the app, merged back from Lakebase |
| `historical_triage_decisions` | Past analyst decisions (model training data, FP rates) |
| `customers` | Customer metadata (join on `customer_id`) |

Genie runs queries as the asking user, so the Unity Catalog row filter on `gold_enriched_alerts`
applies: analysts only see alert rows for their assigned customers. **Scope note:** the row filter is on
the alerts table only — `gold_alert_predictions`, `gold_investigation_summaries` and `analyst_actions`
are not filtered, so this space is intended for SOC managers; per-analyst spaces would need filters on
those tables too (e.g. by adding `customer_id` and the same filter function).

In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
from databricks.sdk import WorkspaceClient

GENIE_SPACE_ID = "01f1c10367b81930a4413866595bf726"

w = WorkspaceClient()
space = w.api_client.do("GET", f"/api/2.0/genie/spaces/{GENIE_SPACE_ID}")
print(f"✅ Genie space: {space['title']}")
print(f"   {space['description']}")
print(f"   URL: {w.config.host}/genie/rooms/{GENIE_SPACE_ID}")

✅ Genie space: SOC Alert Triage
   Ask questions in plain English about security alerts, ML triage predictions, AI investigation summaries, analyst performance and false-positive rates across MDR customers (synthetic data, Sep 2026).
   URL: https://<workspace-host>/genie/rooms/01f1c10367b81930a4413866595bf726


In [0]:
questions = [
    "What share of real threats did the model prioritize as critical or high?",
    "Which 5 customers have the most critical alerts?",
    "What is our historical false positive rate by detection source?",
    "How much analyst workload can the ML model remove?",
]

import time

API = f"/api/2.0/genie/spaces/{GENIE_SPACE_ID}"

def ask_genie(question, timeout_s=180):
    start = w.api_client.do("POST", f"{API}/start-conversation", body={"content": question})
    cid, mid = start["conversation_id"], start["message_id"]
    deadline = time.time() + timeout_s
    while True:
        msg = w.api_client.do("GET", f"{API}/conversations/{cid}/messages/{mid}")
        if msg["status"] in ("COMPLETED", "FAILED", "CANCELLED") or time.time() > deadline:
            return cid, mid, msg
        time.sleep(3)

for q in questions:
    cid, mid, msg = ask_genie(q)
    print("=" * 90)
    print(f"❓ {q}   [{msg['status']}]")
    for att in msg.get("attachments", []):
        if "query" in att:
            print(f"\n🧮 Genie SQL:\n{att['query']['query']}")
            result = w.api_client.do(
                "GET", f"{API}/conversations/{cid}/messages/{mid}/attachments/{att['attachment_id']}/query-result")
            sr = result.get("statement_response", {})
            rows = sr.get("result", {}).get("data_array") or []
            if rows:
                cols = [c["name"] for c in sr["manifest"]["schema"]["columns"]]
                print("\n📊 Result (first 10 rows):")
                print(" | ".join(cols))
                for row in rows[:10]:
                    print(" | ".join("" if v is None else str(v) for v in row))
        if "text" in att:
            print(f"\n💬 Genie answer:\n{att['text']['content']}")

❓ What share of real threats did the model prioritize as critical or high?   [COMPLETED]

🧮 Genie SQL:
SELECT
  `e`.`source_type`,
  COUNT(*) AS `real_threats`,
  SUM(CASE WHEN `p`.`ml_predicted_severity` IN ('critical', 'high') THEN 1 ELSE 0 END) AS `prioritized`,
  ROUND(try_divide(100.0 * SUM(CASE WHEN `p`.`ml_predicted_severity` IN ('critical', 'high') THEN 1 ELSE 0 END),COUNT(*)), 1) AS `pct_prioritized`
FROM `rajarora_febar`.`soc_alert_triage`.`gold_enriched_alerts` `e`
JOIN `rajarora_febar`.`soc_alert_triage`.`gold_alert_predictions` `p`
  ON `e`.`event_id` = `p`.`event_id`
WHERE `e`.`is_true_positive` = TRUE
  AND `e`.`source_type` IS NOT NULL
GROUP BY `e`.`source_type`
ORDER BY `pct_prioritized` DESC

📊 Result (first 10 rows):
source_type | real_threats | prioritized | pct_prioritized
cloud_audit | 2444 | 2406 | 98.4
firewall | 8902 | 8645 | 97.1
auth | 6292 | 6063 | 96.4
edr | 4894 | 3523 | 72.0
email | 2485 | 1762 | 70.9

💬 Genie answer:
Among real threats, the model most of

In [0]:
%sql
-- Benchmark: what share of ground-truth threats did the model put in the top two tiers?
SELECT
    COUNT(*) AS real_threats,
    SUM(CASE WHEN p.ml_predicted_severity IN ('critical', 'high') THEN 1 ELSE 0 END) AS prioritized,
    ROUND(AVG(CASE WHEN p.ml_predicted_severity IN ('critical', 'high') THEN 1 ELSE 0 END) * 100, 1) AS pct_prioritized
FROM rajarora_febar.soc_alert_triage.gold_enriched_alerts e
JOIN rajarora_febar.soc_alert_triage.gold_alert_predictions p ON e.event_id = p.event_id
WHERE e.is_true_positive

real_threats,prioritized,pct_prioritized
25017,22399,89.5


In [0]:
%sql
SELECT customer_id, customer_name, industry, tier,
       COUNT(*) AS total_alerts,
       SUM(CASE WHEN risk_tier = 'critical' THEN 1 ELSE 0 END) AS critical_alerts,
       ROUND(AVG(composite_risk_score), 2) AS avg_risk_score
FROM rajarora_febar.soc_alert_triage.gold_enriched_alerts
GROUP BY customer_id, customer_name, industry, tier
ORDER BY critical_alerts DESC
LIMIT 10

customer_id,customer_name,industry,tier,total_alerts,critical_alerts,avg_risk_score
CUST-0052,Summit Solutions,Retail,Mid-Market,5016,141,3.69
CUST-0064,Vertex Partners,Legal,Enterprise,5144,138,3.76
CUST-0078,Apex Inc,Government,SMB,5122,138,3.71
CUST-0091,Horizon Solutions,Logistics,Enterprise,4921,135,3.66
CUST-0079,Horizon Holdings,Energy,SMB,5096,135,3.73
CUST-0053,Acme LLC,Retail,Enterprise,4964,134,3.83
CUST-0044,Summit Corp,Healthcare,Enterprise,4994,133,3.7
CUST-0093,Global Partners,Logistics,Enterprise,4978,132,3.8
CUST-0009,Global Systems,Technology,SMB,5092,132,3.78
CUST-0083,Pacific Partners,Education,Enterprise,5021,132,3.68


In [0]:
%sql
SELECT analyst_id,
       COUNT(*) AS actions,
       ROUND(AVG(time_to_triage_seconds) / 60, 1) AS avg_triage_minutes,
       ROUND(PERCENTILE(time_to_triage_seconds, 0.95) / 60, 1) AS p95_triage_minutes,
       SUM(CASE WHEN action_type = 'escalate' THEN 1 ELSE 0 END) AS escalations,
       SUM(CASE WHEN action_type = 'false_positive' THEN 1 ELSE 0 END) AS false_positives
FROM rajarora_febar.soc_alert_triage.analyst_actions
GROUP BY analyst_id
ORDER BY avg_triage_minutes

analyst_id,actions,avg_triage_minutes,p95_triage_minutes,escalations,false_positives
analyst_15,59,17.7,40.7,16,7
analyst_1,34,19.4,40.1,4,8
analyst_7,38,21.4,41.4,4,3
analyst_10,61,22.1,43.0,14,12
analyst_2,39,22.2,39.8,7,6
analyst_20,51,22.3,43.4,15,14
analyst_6,46,22.6,42.9,9,10
analyst_4,51,22.6,44.0,13,13
analyst_13,57,22.6,42.9,10,12
analyst_18,56,22.8,44.2,14,7


In [0]:
%sql
SELECT source_type,
       COUNT(*) AS triaged_alerts,
       ROUND(AVG(CASE WHEN triage_outcome IN ('false_positive', 'benign') THEN 1 ELSE 0 END) * 100, 1) AS fp_or_benign_pct,
       ROUND(AVG(CASE WHEN triage_outcome = 'true_positive' THEN 1 ELSE 0 END) * 100, 1) AS true_positive_pct
FROM rajarora_febar.soc_alert_triage.historical_triage_decisions
GROUP BY source_type
ORDER BY fp_or_benign_pct DESC

source_type,triaged_alerts,fp_or_benign_pct,true_positive_pct
auth,10081,70.3,14.7
email,9898,70.1,14.9
cloud_audit,10012,69.9,15.0
firewall,9970,69.8,14.8
edr,10039,69.6,15.4


## ✅ Genie Space Ready

### Sample questions
- "Which customers have the most critical alerts?"
- "How much analyst workload can the ML model remove?"
- "What share of real threats did the model prioritize?"
- "What is our false positive rate by detection source?"
- "Show the daily alert trend for critical and high alerts"

**Next step**: Open the analyst workbench (Databricks App deployed from `/app`)